# NVDA 주가 방향 예측 — 최종 모델링

## 피처 구성

| 범주 | 피처 | shift |
|------|------|-------|
| NVDA 기술지표 | Return_1D, MA20_ratio, Volume_ratio, RSI, MACD | O |
| 상대강도 | NVDA_vs_QQQ, NVDA_vs_SOX | O |
| 변동성 | NVDA_RealVol_20d, VIX, VIX_delta | O |
| 섹터/종목 | SOX_Return, TSM_Return, QQQ_Return, MSFT_Return, META_Return | O |
| 매크로 | US_10Y_Yield, DXY_Return, US_CPI | O |
| 이벤트 캘린더 | is_nvda_post_earnings, is_nvda_earnings_eve, is_fomc_day, is_cpi_day | X |

**캘린더 피처는 사전에 알려진 일정이므로 shift 불필요** (시장 데이터는 lookahead 방지를 위해 shift(1) 적용)

**타겟 정의**
```
y_t = 1  if NVDA_Close_t > NVDA_Close_{t-1}
X_t = 시장 피처는 t-1일 데이터 + 캘린더 피처는 t일
```

In [ ]:
# 분석일: 2026-06-03
import os
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import ta
import torch
import torch.nn as nn
import yfinance as yf
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset
from xgboost import XGBClassifier

np.random.seed(42)
torch.manual_seed(42)

RAW_DIR       = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
MODEL_DIR     = Path("../models")
RESULTS_DIR   = Path("../reports/results")
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

FORCE_RETRAIN = False
EPOCHS, BATCH_SIZE, WINDOW = 150, 32, 20
print("imports OK")

## 1. 이벤트 캘린더 구성

In [ ]:
# ── NVDA 실적 발표일 (yfinance 기반, 장외 발표) ──────────────────────────────
nvda_ticker = yf.Ticker("NVDA")
earnings_raw = nvda_ticker.earnings_dates
earnings_dates = pd.DatetimeIndex(
    earnings_raw.index.tz_localize(None) if earnings_raw.index.tzinfo is None
    else earnings_raw.index.tz_convert(None)
).normalize()
print(f"NVDA 실적 발표일: {sorted(earnings_dates)[:5]} ...")

# ── FOMC 결정 발표일 (하드코딩: 2020~2025 실제 일정) ─────────────────────────
FOMC_DATES = pd.to_datetime([
    # 2020
    "2020-01-29", "2020-03-03", "2020-03-15", "2020-04-29",
    "2020-06-10", "2020-07-29", "2020-09-16", "2020-11-05", "2020-12-16",
    # 2021
    "2021-01-27", "2021-03-17", "2021-04-28", "2021-06-16",
    "2021-07-28", "2021-09-22", "2021-11-03", "2021-12-15",
    # 2022
    "2022-01-26", "2022-03-16", "2022-05-04", "2022-06-15",
    "2022-07-27", "2022-09-21", "2022-11-02", "2022-12-14",
    # 2023
    "2023-02-01", "2023-03-22", "2023-05-03", "2023-06-14",
    "2023-07-26", "2023-09-20", "2023-11-01", "2023-12-13",
    # 2024
    "2024-01-31", "2024-03-20", "2024-05-01", "2024-06-12",
    "2024-07-31", "2024-09-18", "2024-11-07", "2024-12-18",
    # 2025
    "2025-01-29", "2025-03-19", "2025-05-07",
])
print(f"FOMC 날짜 수: {len(FOMC_DATES)}개")

## 2. 피처 엔지니어링

In [ ]:
prices = pd.read_csv(RAW_DIR / "prices_raw.csv", index_col=0, parse_dates=True)
cpi    = pd.read_csv(RAW_DIR / "cpi_raw.csv",    index_col=0, parse_dates=True)

nvda_raw = yf.download("NVDA", start="2019-12-01", end="2025-05-23",
                        auto_adjust=True, progress=False)
prices["NVDA_Volume"] = nvda_raw["Volume"][nvda_raw.index.isin(prices.index)]

nvda = prices["NVDA"]

# ── 기존 피처 ──────────────────────────────────────────────────────────────────
prices["NVDA_Return_1D"]    = nvda.pct_change()
prices["NVDA_MA20_ratio"]   = (nvda - nvda.rolling(20).mean()) / nvda.rolling(20).mean()
prices["NVDA_Volume_ratio"] = prices["NVDA_Volume"] / prices["NVDA_Volume"].rolling(20).mean()
prices["NVDA_RSI"]          = ta.momentum.RSIIndicator(close=nvda, window=14).rsi()
prices["NVDA_MACD"]         = ta.trend.MACD(close=nvda).macd()

for col in ["SOX", "TSM", "QQQ", "DXY", "MSFT", "META"]:
    prices[f"{col}_Return"] = prices[col].pct_change()

prices["NVDA_vs_QQQ"] = prices["NVDA_Return_1D"] - prices["QQQ_Return"]
prices["NVDA_vs_SOX"] = prices["NVDA_Return_1D"] - prices["SOX_Return"]

prices["US_CPI"] = cpi.reindex(prices.index, method="ffill")["US_CPI"]
prices["target"] = (nvda > nvda.shift(1)).astype(int)

# ── 신규 시장 피처 (shift 대상) ───────────────────────────────────────────────
prices["NVDA_RealVol_20d"] = prices["NVDA_Return_1D"].rolling(20).std()  # 20일 실현 변동성
prices["VIX_delta"]        = prices["VIX"].diff()                         # VIX 전일 대비 변화량

MARKET_FEATURES = [
    "NVDA_Return_1D", "NVDA_MA20_ratio", "NVDA_Volume_ratio", "NVDA_RSI", "NVDA_MACD",
    "NVDA_vs_QQQ", "NVDA_vs_SOX",
    "NVDA_RealVol_20d", "VIX_delta",     # 신규
    "SOX_Return", "TSM_Return",
    "US_10Y_Yield", "QQQ_Return", "DXY_Return", "US_CPI",
    "VIX",
    "MSFT_Return", "META_Return",
]

# 시장 피처: shift(1) 적용 (lookahead 방지)
df = prices[MARKET_FEATURES].shift(1).copy()
df["target"] = prices["target"]
df.dropna(inplace=True)
df = df[df.index >= "2020-01-02"]

# ── 신규 캘린더 피처 (shift 불필요 — 사전에 알려진 일정) ─────────────────────
trading_days = df.index

# NVDA 실적: 장외 발표 → 다음 거래일이 반응일
post_earnings_days = set()
earnings_eve_days  = set()
for ed in earnings_dates:
    # 다음 거래일 찾기
    future = trading_days[trading_days > ed]
    if len(future) > 0:
        post_earnings_days.add(future[0])
    # 전일 거래일 찾기
    prior = trading_days[trading_days < ed]
    if len(prior) > 0:
        earnings_eve_days.add(prior[-1])

df["is_nvda_post_earnings"] = df.index.isin(post_earnings_days).astype(int)
df["is_nvda_earnings_eve"]  = df.index.isin(earnings_eve_days).astype(int)

# FOMC: 결정 발표일 당일 (장 중 14:00 발표)
df["is_fomc_day"] = df.index.isin(FOMC_DATES).astype(int)

# CPI: US_CPI 값이 바뀌는 날 = 새 CPI 발표일
# prices에서 CPI 변화 감지 (원본 인덱스 기준)
cpi_change_mask = prices["US_CPI"] != prices["US_CPI"].shift(1)
cpi_release_days = prices.index[cpi_change_mask & prices.index.isin(trading_days)]
df["is_cpi_day"] = df.index.isin(cpi_release_days).astype(int)

CALENDAR_FEATURES = [
    "is_nvda_post_earnings", "is_nvda_earnings_eve", "is_fomc_day", "is_cpi_day",
]
FEATURE_COLS = MARKET_FEATURES + CALENDAR_FEATURES

df.to_csv(PROCESSED_DIR / "features.csv")
print(f"피처 수: {len(FEATURE_COLS)}개")
print(f"데이터: {df.shape[0]}행  ({df.index.min().date()} ~ {df.index.max().date()})")
print(f"타겟 상승: {df['target'].mean()*100:.1f}%")
print()
print("캘린더 피처 분포:")
for c in CALENDAR_FEATURES:
    n = df[c].sum()
    print(f"  {c}: {n}일 ({n/len(df)*100:.1f}%)")

## 3. 데이터 분리

In [ ]:
train = df[df.index <= "2023-06-30"]
val   = df[(df.index >= "2023-07-01") & (df.index <= "2024-06-30")]
test  = df[df.index >= "2024-07-01"]

train.to_csv(PROCESSED_DIR / "train.csv")
val.to_csv(PROCESSED_DIR / "val.csv")
test.to_csv(PROCESSED_DIR / "test.csv")

X_train, y_train = train[FEATURE_COLS].values, train["target"].values
X_val,   y_val   = val[FEATURE_COLS].values,   val["target"].values
X_test,  y_test  = test[FEATURE_COLS].values,  test["target"].values

majority = max(y_test.mean(), 1 - y_test.mean()) * 100
for name, s in [("Train", train), ("Val", val), ("Test", test)]:
    print(f"{name}: {len(s)}일  상승 {s['target'].mean()*100:.1f}%")
print(f"\nTest Majority baseline: {majority:.1f}%")

# 테스트 기간 이벤트 수 확인
print("\n[Test 기간 이벤트]")
for c in CALENDAR_FEATURES:
    print(f"  {c}: {test[c].sum()}일")

## 4. 공통 함수

In [ ]:
def evaluate(name, y_true, y_pred, y_prob):
    """모델 성능 지표를 딕셔너리로 반환한다."""
    return {
        "model":     name,
        "accuracy":  round(accuracy_score(y_true, y_pred) * 100, 2),
        "precision": round(precision_score(y_true, y_pred, zero_division=0), 4),
        "recall":    round(recall_score(y_true, y_pred, zero_division=0), 4),
        "f1":        round(f1_score(y_true, y_pred, zero_division=0), 4),
        "roc_auc":   round(roc_auc_score(y_true, y_prob), 4),
    }

def save_preds(name, idx, y_true, y_pred, y_prob):
    """예측 결과를 CSV로 저장한다."""
    pd.DataFrame({"y_true": y_true, "y_pred": y_pred, "y_prob": y_prob}, index=idx
                 ).to_csv(RESULTS_DIR / f"{name}_test_predictions.csv")

def best_threshold(y_true, y_prob):
    """Val 기준 F1 최대 임계값을 탐색한다."""
    best_t, best_f1 = 0.5, 0
    for t in np.arange(0.3, 0.71, 0.01):
        f1 = f1_score(y_true, (y_prob > t).astype(int), zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return round(best_t, 2)

all_metrics   = []
all_probs_val = {}   # 고확신 필터 분석용
all_probs_test = {}
print("준비 완료")

## 5. Logistic Regression

In [ ]:
lr_path = MODEL_DIR / "lr_model.joblib"

if lr_path.exists() and not FORCE_RETRAIN:
    lr_pipe = joblib.load(lr_path)
    print("저장된 모델 로드")
else:
    best_f1, best_c, best_pipe = 0, 1, None
    for C in [0.001, 0.01, 0.1, 1, 10, 100]:
        pipe = Pipeline([("scaler", StandardScaler()),
                         ("model", LogisticRegression(C=C, max_iter=1000, random_state=42))])
        pipe.fit(X_train, y_train)
        f1 = f1_score(y_val, pipe.predict(X_val), zero_division=0)
        print(f"  C={C:<6} val F1={f1:.4f}")
        if f1 > best_f1:
            best_f1, best_c, best_pipe = f1, C, pipe
    lr_pipe = best_pipe
    joblib.dump(lr_pipe, lr_path)
    print(f"\n최적 C={best_c}")

t = best_threshold(y_val, lr_pipe.predict_proba(X_val)[:, 1])
prob_v = lr_pipe.predict_proba(X_val)[:, 1]
prob   = lr_pipe.predict_proba(X_test)[:, 1]
pred   = (prob > t).astype(int)
save_preds("lr", test.index, y_test, pred, prob)
all_probs_val["LR"]  = prob_v
all_probs_test["LR"] = prob
m = evaluate("LR", y_test, pred, prob)
all_metrics.append(m)
print(f"임계값={t}  Accuracy={m['accuracy']}%  F1={m['f1']}  ROC-AUC={m['roc_auc']}")

## 6. XGBoost

In [ ]:
xgb_path = MODEL_DIR / "xgb_model.joblib"

if xgb_path.exists() and not FORCE_RETRAIN:
    xgb_model = joblib.load(xgb_path)
    print("저장된 모델 로드")
else:
    best_f1, best_params, best_model = 0, {}, None
    for max_depth in [2, 3]:
        for lr in [0.01, 0.05]:
            for n_est in [100, 200]:
                model = XGBClassifier(
                    max_depth=max_depth,
                    learning_rate=lr,
                    n_estimators=n_est,
                    subsample=0.7,
                    colsample_bytree=0.7,
                    min_child_weight=5,
                    reg_alpha=0.1,
                    reg_lambda=1.5,
                    random_state=42,
                    verbosity=0,
                )
                model.fit(X_train, y_train)
                f1 = f1_score(y_val, model.predict(X_val), zero_division=0)
                if f1 > best_f1:
                    best_f1 = f1
                    best_params = {"max_depth": max_depth, "lr": lr, "n_est": n_est}
                    best_model = model
    xgb_model = best_model
    joblib.dump(xgb_model, xgb_path)
    print(f"최적 {best_params}  val F1={best_f1:.4f}")

t = best_threshold(y_val, xgb_model.predict_proba(X_val)[:, 1])
prob_v = xgb_model.predict_proba(X_val)[:, 1]
prob   = xgb_model.predict_proba(X_test)[:, 1]
pred   = (prob > t).astype(int)
save_preds("xgb", test.index, y_test, pred, prob)
all_probs_val["XGBoost"]  = prob_v
all_probs_test["XGBoost"] = prob
m = evaluate("XGBoost", y_test, pred, prob)
all_metrics.append(m)
print(f"임계값={t}  Accuracy={m['accuracy']}%  F1={m['f1']}  ROC-AUC={m['roc_auc']}")

# 피처 중요도 (상위 10개)
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

fi = pd.Series(xgb_model.feature_importances_, index=FEATURE_COLS).sort_values(ascending=False)
print("\nXGBoost 피처 중요도 상위 10:")
print(fi.head(10).round(4).to_string())

## 7. MLP

In [ ]:
pos_weight = torch.tensor([3.0])
scaler_mlp = StandardScaler()
X_tr = scaler_mlp.fit_transform(X_train)
X_va = scaler_mlp.transform(X_val)
X_te = scaler_mlp.transform(X_test)
joblib.dump(scaler_mlp, MODEL_DIR / "mlp_scaler.joblib")

class MLP(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d, 64), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, 32), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(32, 1))
    def forward(self, x): return self.net(x).squeeze(1)

mlp_path = MODEL_DIR / "mlp_model.pt"
if mlp_path.exists() and not FORCE_RETRAIN:
    mlp = MLP(X_tr.shape[1])
    mlp.load_state_dict(torch.load(mlp_path, weights_only=True))
    print("저장된 모델 로드")
else:
    mlp  = MLP(X_tr.shape[1])
    opt  = torch.optim.Adam(mlp.parameters(), lr=1e-3, weight_decay=1e-4)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    loader = DataLoader(
        TensorDataset(torch.FloatTensor(X_tr), torch.FloatTensor(y_train.astype(float))),
        batch_size=BATCH_SIZE, shuffle=True)
    best_f1, best_state = 0, None
    for ep in range(EPOCHS):
        mlp.train()
        for xb, yb in loader:
            opt.zero_grad(); crit(mlp(xb), yb).backward(); opt.step()
        mlp.eval()
        with torch.no_grad():
            pv = (torch.sigmoid(mlp(torch.FloatTensor(X_va))) > 0.5).numpy().astype(int)
            f1 = f1_score(y_val, pv, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_state = {k: v.clone() for k, v in mlp.state_dict().items()}
    mlp.load_state_dict(best_state)
    torch.save(mlp.state_dict(), mlp_path)
    print(f"학습 완료  val F1={best_f1:.4f}")

mlp.eval()
with torch.no_grad():
    prob_v = torch.sigmoid(mlp(torch.FloatTensor(X_va))).numpy()
    prob_t = torch.sigmoid(mlp(torch.FloatTensor(X_te))).numpy()
t = best_threshold(y_val, prob_v)
pred = (prob_t > t).astype(int)
save_preds("mlp", test.index, y_test, pred, prob_t)
all_probs_val["MLP"]  = prob_v
all_probs_test["MLP"] = prob_t
m = evaluate("MLP", y_test, pred, prob_t)
all_metrics.append(m)
print(f"임계값={t}  Accuracy={m['accuracy']}%  F1={m['f1']}  ROC-AUC={m['roc_auc']}")

## 8. Dilated TCN

In [ ]:
scaler_tcn = StandardScaler()
X_tr2 = scaler_tcn.fit_transform(X_train)
X_va2 = scaler_tcn.transform(X_val)
X_te2 = scaler_tcn.transform(X_test)
joblib.dump(scaler_tcn, MODEL_DIR / "tcn_scaler.joblib")

def make_seq(X, y, w):
    """슬라이딩 윈도우로 시퀀스를 생성한다."""
    xs, ys = [], []
    for i in range(w, len(X)):
        xs.append(X[i-w:i]); ys.append(y[i])
    return np.array(xs), np.array(ys)

X_tr_s, y_tr_s = make_seq(X_tr2, y_train, WINDOW)
X_va_s, y_va_s = make_seq(X_va2, y_val,   WINDOW)
X_te_s, y_te_s = make_seq(X_te2, y_test,  WINDOW)
test_idx_tcn = test.index[WINDOW:]

class DilatedTCN(nn.Module):
    def __init__(self, d, ch=32):
        super().__init__()
        layers, in_ch = [], d
        for dil in [1, 2, 4, 8]:
            layers += [nn.Conv1d(in_ch, ch, 3, padding=dil, dilation=dil),
                       nn.BatchNorm1d(ch), nn.ReLU(), nn.Dropout(0.2)]
            in_ch = ch
        self.tcn = nn.Sequential(*layers)
        self.fc  = nn.Linear(ch, 1)
    def forward(self, x):
        x = x.permute(0, 2, 1)
        return self.fc(self.tcn(x)[:, :, -1]).squeeze(1)

tcn_path = MODEL_DIR / "tcn_model.pt"
if tcn_path.exists() and not FORCE_RETRAIN:
    tcn = DilatedTCN(X_tr_s.shape[2])
    tcn.load_state_dict(torch.load(tcn_path, weights_only=True))
    print("저장된 모델 로드")
else:
    tcn  = DilatedTCN(X_tr_s.shape[2])
    opt  = torch.optim.Adam(tcn.parameters(), lr=1e-3, weight_decay=1e-4)
    crit = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    loader = DataLoader(
        TensorDataset(torch.FloatTensor(X_tr_s), torch.FloatTensor(y_tr_s.astype(float))),
        batch_size=BATCH_SIZE, shuffle=True)
    best_f1, best_state = 0, None
    for ep in range(EPOCHS):
        tcn.train()
        for xb, yb in loader:
            opt.zero_grad(); crit(tcn(xb), yb).backward(); opt.step()
        tcn.eval()
        with torch.no_grad():
            pv = (torch.sigmoid(tcn(torch.FloatTensor(X_va_s))) > 0.5).numpy().astype(int)
            f1 = f1_score(y_va_s, pv, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_state = {k: v.clone() for k, v in tcn.state_dict().items()}
    tcn.load_state_dict(best_state)
    torch.save(tcn.state_dict(), tcn_path)
    print(f"학습 완료  val F1={best_f1:.4f}")

tcn.eval()
with torch.no_grad():
    prob_v = torch.sigmoid(tcn(torch.FloatTensor(X_va_s))).numpy()
    prob_t = torch.sigmoid(tcn(torch.FloatTensor(X_te_s))).numpy()
t = best_threshold(y_va_s, prob_v)
pred = (prob_t > t).astype(int)
save_preds("tcn", test_idx_tcn, y_te_s, pred, prob_t)
all_probs_val["DilatedTCN"]  = prob_v
all_probs_test["DilatedTCN"] = prob_t
m = evaluate("DilatedTCN", y_te_s, pred, prob_t)
all_metrics.append(m)
print(f"임계값={t}  Accuracy={m['accuracy']}%  F1={m['f1']}  ROC-AUC={m['roc_auc']}")

## 9. TCN 재설계 실험

기존 TCN은 `pos_weight=3.0`, Validation F1 기준 epoch 선택, 캘린더 dummy 포함으로 상승 예측에 치우쳤다. 재설계 실험은 시장 피처만 사용하고, `pos_weight`를 제거하거나 조정하며, best epoch를 Validation ROC-AUC 기준으로 선택한다.


In [ ]:
from sklearn.metrics import balanced_accuracy_score, matthews_corrcoef

MARKET_ONLY_FEATURES = [c for c in FEATURE_COLS if c not in CALENDAR_FEATURES]

def best_threshold_bal_acc(y_true, y_prob):
    """Validation balanced accuracy가 최대인 threshold를 선택한다."""
    best_t, best_s = 0.5, -1
    for t in np.arange(0.30, 0.71, 0.01):
        pred = (y_prob >= t).astype(int)
        s = balanced_accuracy_score(y_true, pred)
        if s > best_s:
            best_s, best_t = s, t
    return round(float(best_t), 2), round(float(best_s), 4)

def run_tcn_redesign(name, feature_cols, pos_mode="none"):
    """TCN 재설계 비교 실험: Validation ROC-AUC로 best epoch를 선택한다."""
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(train[feature_cols].values)
    X_va = scaler.transform(val[feature_cols].values)
    X_te = scaler.transform(test[feature_cols].values)

    X_tr_s, y_tr_s = make_seq(X_tr, y_train, WINDOW)
    X_va_s, y_va_s = make_seq(X_va, y_val, WINDOW)
    X_te_s, y_te_s = make_seq(X_te, y_test, WINDOW)

    model = DilatedTCN(X_tr_s.shape[2])
    opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    if pos_mode == "none":
        crit = nn.BCEWithLogitsLoss()
        pos_weight_value = 1.0
    elif pos_mode == "balanced":
        n_pos = y_tr_s.sum()
        n_neg = len(y_tr_s) - n_pos
        pos_weight_value = float(n_neg / n_pos)
        crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight_value], dtype=torch.float32))
    elif pos_mode == "three":
        pos_weight_value = 3.0
        crit = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([3.0], dtype=torch.float32))
    else:
        raise ValueError(pos_mode)

    loader = DataLoader(
        TensorDataset(torch.FloatTensor(X_tr_s), torch.FloatTensor(y_tr_s.astype(float))),
        batch_size=BATCH_SIZE,
        shuffle=True,
    )
    best_auc, best_state, best_epoch = -1, None, 0
    for ep in range(1, EPOCHS + 1):
        model.train()
        for xb, yb in loader:
            opt.zero_grad()
            crit(model(xb), yb).backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            prob_v = torch.sigmoid(model(torch.FloatTensor(X_va_s))).numpy()
        auc = roc_auc_score(y_va_s, prob_v)
        if auc > best_auc:
            best_auc = auc
            best_epoch = ep
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        prob_v = torch.sigmoid(model(torch.FloatTensor(X_va_s))).numpy()
        prob_t = torch.sigmoid(model(torch.FloatTensor(X_te_s))).numpy()

    threshold, val_bal_acc = best_threshold_bal_acc(y_va_s, prob_v)
    pred_t = (prob_t >= threshold).astype(int)
    aligned_majority = max(y_te_s.mean(), 1 - y_te_s.mean()) * 100
    return {
        "variant": name,
        "features": len(feature_cols),
        "calendar_features": int(any(c in feature_cols for c in CALENDAR_FEATURES)),
        "pos_mode": pos_mode,
        "pos_weight": round(pos_weight_value, 4),
        "best_epoch": best_epoch,
        "val_roc_auc": round(roc_auc_score(y_va_s, prob_v), 4),
        "val_threshold_bal_acc": threshold,
        "val_bal_acc_at_threshold": val_bal_acc,
        "test_n": len(y_te_s),
        "test_aligned_majority": round(aligned_majority, 2),
        "test_accuracy": round(accuracy_score(y_te_s, pred_t) * 100, 2),
        "test_balanced_accuracy": round(balanced_accuracy_score(y_te_s, pred_t), 4),
        "test_precision": round(precision_score(y_te_s, pred_t, zero_division=0), 4),
        "test_recall": round(recall_score(y_te_s, pred_t, zero_division=0), 4),
        "test_f1": round(f1_score(y_te_s, pred_t, zero_division=0), 4),
        "test_mcc": round(matthews_corrcoef(y_te_s, pred_t), 4),
        "test_roc_auc": round(roc_auc_score(y_te_s, prob_t), 4),
        "positive_rate": round(pred_t.mean(), 4),
    }

redesign_rows = []
for feature_name, feature_cols in [("all_features", FEATURE_COLS), ("market_only", MARKET_ONLY_FEATURES)]:
    for pos_mode in ["none", "balanced", "three"]:
        redesign_rows.append(run_tcn_redesign(feature_name, feature_cols, pos_mode))

tcn_redesign_df = pd.DataFrame(redesign_rows).sort_values(
    ["test_roc_auc", "test_balanced_accuracy"], ascending=False
)
tcn_redesign_df.to_csv(RESULTS_DIR / "tcn_redesign_metrics.csv", index=False)
print(tcn_redesign_df.to_string(index=False))


## 10. 고확신 필터 재검증

고확신 threshold는 test를 보고 고르지 않고, validation precision 기준으로 선택한 뒤 test에 1회 적용한다.


In [ ]:
import matplotlib.pyplot as plt
plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False

xgb_prob_val = all_probs_val["XGBoost"]
xgb_prob_test = all_probs_test["XGBoost"]
thresholds = np.arange(0.45, 0.61, 0.01)
results_filter = []

for thr in thresholds:
    val_mask = xgb_prob_val >= thr
    test_mask = xgb_prob_test >= thr
    results_filter.append({
        "threshold": round(float(thr), 2),
        "val_trades": int(val_mask.sum()),
        "val_coverage": round(val_mask.mean() * 100, 1),
        "val_precision": round(y_val[val_mask].mean(), 4) if val_mask.any() else np.nan,
        "test_trades": int(test_mask.sum()),
        "test_coverage": round(test_mask.mean() * 100, 1),
        "test_precision": round(y_test[test_mask].mean(), 4) if test_mask.any() else np.nan,
    })

filter_df = pd.DataFrame(results_filter)
filter_df.to_csv(RESULTS_DIR / "high_confidence_filter_val_selected.csv", index=False)

candidates = filter_df[filter_df["val_coverage"] >= 50].copy()
best_row = candidates.sort_values(["val_precision", "val_coverage"], ascending=[False, False]).iloc[0]

print("XGBoost validation 기준 고확신 필터:")
print(filter_df.to_string(index=False))
print(
    f"\nValidation 선택 threshold={best_row['threshold']}  "
    f"val_precision={best_row['val_precision']}  val_coverage={best_row['val_coverage']}%  "
    f"test_precision={best_row['test_precision']}  test_coverage={best_row['test_coverage']}%"
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(filter_df["threshold"], filter_df["val_precision"], marker="o", label="Val precision")
ax1.plot(filter_df["threshold"], filter_df["test_precision"], marker="o", label="Test precision")
ax1.axvline(best_row["threshold"], color="gray", linestyle="--", label="Val-selected threshold")
ax1.set_xlabel("확률 임계값")
ax1.set_ylabel("Precision")
ax1.set_title("고확신 필터: validation 기준 threshold 선택")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(filter_df["threshold"], filter_df["val_coverage"], marker="s", label="Val coverage")
ax2.plot(filter_df["threshold"], filter_df["test_coverage"], marker="s", label="Test coverage")
ax2.axvline(best_row["threshold"], color="gray", linestyle="--")
ax2.set_xlabel("확률 임계값")
ax2.set_ylabel("커버리지 (%)")
ax2.set_title("고확신 필터: 커버리지")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "high_confidence_filter_val_selected.png", dpi=150, bbox_inches="tight")
plt.show()


## 10. 결과 비교

In [ ]:
metrics_df = pd.DataFrame(all_metrics)
metrics_df.to_csv(RESULTS_DIR / "model_metrics.csv", index=False)

print("=" * 55)
print("모델 성능 비교 (Test Set)")
print("=" * 55)
print(metrics_df.to_string(index=False))
print(f"
Majority baseline: {majority:.1f}%")

# 모델 성능 비교 차트
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
metrics_keys = [("accuracy", "Accuracy (%)"), ("roc_auc", "ROC-AUC"), ("f1", "F1")]
for ax, (metric, label) in zip(axes, metrics_keys):
    x = np.arange(len(all_metrics))
    model_names = [m["model"] for m in all_metrics]
    vals = [m[metric] for m in all_metrics]
    ax.bar(x, vals, 0.5, color="#DD8452", alpha=0.8)
    ax.set_xticks(x)
    ax.set_xticklabels(model_names, rotation=15)
    ax.set_title(label)
    ax.grid(True, alpha=0.3, axis="y")

plt.suptitle("모델 성능 비교 (Test Set)", fontsize=13)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "model_comparison.png", dpi=150, bbox_inches="tight")
plt.show()

## 11. 이벤트별 수익률 분포 검증

이벤트 더미가 실제로 NVDA 방향성과 관련이 있는지 사후 검증한다.

In [ ]:
from scipy import stats

# 전체 데이터(train+val+test)에서 이벤트별 실제 수익률 분포 확인
nvda_ret = prices["NVDA"].pct_change().rename("NVDA_Return")
event_check = pd.DataFrame(index=df.index)
event_check["NVDA_Return"] = nvda_ret.reindex(df.index)
for c in CALENDAR_FEATURES:
    event_check[c] = df[c]

print("이벤트별 NVDA 수익률 분포 검증")
print("=" * 55)
for c in CALENDAR_FEATURES:
    event_days = event_check[event_check[c] == 1]["NVDA_Return"].dropna()
    non_event  = event_check[event_check[c] == 0]["NVDA_Return"].dropna()

    t_stat, p_val = stats.ttest_ind(event_days, non_event)
    print(f"\n{c}")
    print(f"  이벤트 평균: {event_days.mean()*100:.3f}%  std: {event_days.std()*100:.3f}%  n={len(event_days)}")
    print(f"  비이벤트 평균: {non_event.mean()*100:.3f}%  std: {non_event.std()*100:.3f}%")
    print(f"  t-test p-value: {p_val:.4f} {'*' if p_val < 0.05 else ''}")

# 시각화
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
colors = ["#4C72B0", "#DD8452"]

for ax, c in zip(axes.flatten(), CALENDAR_FEATURES):
    event_days = event_check[event_check[c] == 1]["NVDA_Return"].dropna() * 100
    non_event  = event_check[event_check[c] == 0]["NVDA_Return"].dropna() * 100

    ax.hist(non_event, bins=50, alpha=0.6, color=colors[0], label=f"비이벤트 (n={len(non_event)})")
    ax.hist(event_days, bins=20, alpha=0.8, color=colors[1], label=f"이벤트 (n={len(event_days)})")
    ax.axvline(non_event.mean(), color=colors[0], linestyle="--", linewidth=1.5)
    ax.axvline(event_days.mean(), color=colors[1], linestyle="--", linewidth=1.5)
    ax.set_title(c)
    ax.set_xlabel("NVDA 수익률 (%)")
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.suptitle("이벤트별 NVDA 수익률 분포", fontsize=13)
plt.tight_layout()
plt.savefig(RESULTS_DIR / "event_return_distribution.png", dpi=150, bbox_inches="tight")
plt.show()